### 1. Install and Configure the Gemini API
To get started, make sure you have added your API key under the Secrets tab (🔑 icon on the left) with the name `GOOGLE_API_KEY`.

In [39]:
# Import SDK and get the API key
from google import genai
from google.genai import types
from google.colab import userdata
from pydantic import BaseModel, Field
from typing import List, Optional
import json
import os

try:
    # Automatically fetches the key securely from the Secrets tab
    GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')
except Exception:
    # Fallback in case of environment setup mismatch
    GOOGLE_API_KEY = os.environ.get('GOOGLE_API_KEY', 'AQ.Ab8RN6JfrunNHjFMrTx3Y966Lz0dSXW4ry2PKuuN1DewJmTHPQ')

client = genai.Client(api_key=GOOGLE_API_KEY)

### 2. Define the Nutrition Schema
We will define a Pydantic model to enforce that the LLM returns structured information suitable for your shopping database.

In [40]:
class Macronutrients(BaseModel):
    carbohydrates_g: float = Field(description="Carbohydrates in grams")
    protein_g: float = Field(description="Protein in grams")
    fat_g: float = Field(description="Fats in grams")

class NutritionInfo(BaseModel):
    product_name: str = Field(description="The standard or recognized name of the food item")
    estimated_calories_kcal: int = Field(description="Estimated calories per standard serving size")
    serving_size: str = Field(description="Standard serving size (e.g., '100g', '1 medium size', '1 cup')")
    macronutrients: Macronutrients
    allergens: List[str] = Field(description="Potential allergens present in the product (e.g., Gluten, Nuts, Dairy, None)")
    health_benefits: List[str] = Field(description="Key health benefits or nutritional highlights of this food item")

### 3. Build the Nutrition Agent Function
Now we create a function that takes a raw food item name or query from the user and returns the structured nutrition facts.

In [41]:
def get_nutrition_facts(food_query: str) -> NutritionInfo:
    prompt = f"Provide detailed nutritional information for the following shopping item: {food_query}"

    response = client.models.generate_content(
        model='gemini-3.5-flash',
        contents=prompt,
        config=types.GenerateContentConfig(
            response_mime_type="application/json",
            response_schema=NutritionInfo,
            system_instruction="You are an expert nutritionist assistant for a grocery shopping application. Provide accurate estimated nutritional details."
        ),
    )
    # Parse back into the Pydantic model
    return NutritionInfo.model_validate_json(response.text)

### 4. Test the Nutrition Agent
Let's test the agent with a few grocery store items!

In [42]:
# Example test 1: Banana
banana_info = get_nutrition_facts("1 medium banana")
print(json.dumps(banana_info.model_dump(), indent=2))

{
  "product_name": "Medium Banana",
  "estimated_calories_kcal": 105,
  "serving_size": "1 medium (118g)",
  "macronutrients": {
    "carbohydrates_g": 27.0,
    "protein_g": 1.3,
    "fat_g": 0.3
  },
  "allergens": [],
  "health_benefits": [
    "Rich in potassium which supports heart health",
    "Good source of vitamin B6 and vitamin C",
    "Contains dietary fiber for healthy digestion",
    "Provides quick natural energy"
  ]
}


In [43]:
# Example test 2: Peanut butter
pb_info = get_nutrition_facts("organic peanut butter")
print(json.dumps(pb_info.model_dump(), indent=2))

{
  "product_name": "Organic Peanut Butter",
  "estimated_calories_kcal": 190,
  "serving_size": "2 tbsp (32g)",
  "macronutrients": {
    "carbohydrates_g": 7.0,
    "protein_g": 8.0,
    "fat_g": 16.0
  },
  "allergens": [
    "Peanuts"
  ],
  "health_benefits": [
    "Good source of plant-based protein",
    "Rich in heart-healthy monounsaturated fats",
    "Contains antioxidant Vitamin E and magnesium"
  ]
}


In [44]:
apple_info = get_nutrition_facts("1 medium apple")
print(json.dumps(apple_info.model_dump(), indent=2))

{
  "product_name": "Medium Apple",
  "estimated_calories_kcal": 95,
  "serving_size": "1 medium (182g)",
  "macronutrients": {
    "carbohydrates_g": 25.1,
    "protein_g": 0.5,
    "fat_g": 0.3
  },
  "allergens": [
    "None"
  ],
  "health_benefits": [
    "High in dietary fiber for digestive health",
    "Good source of Vitamin C",
    "Contains antioxidants supporting heart health"
  ]
}
